## Example of how to process the data

Do the query based on emip_query.json
Output will be stored in src/query/output directory with hierarchical structure based on query parameters

In [1]:
from tools.auxiliary.metadata import metadata_query
from tools.auxiliary import parse_corrected_emip_data

# Have to run this cell 4 times for each expertise level: none, low, medium, high
eye_events_corrected = parse_corrected_emip_data()
query_result = metadata_query(eye_events_corrected)

Possible value for experiment_language are ['Java', 'Scala', 'Python']
Possible value for expertise_experiment_language are ['high', 'medium', 'none', 'low']

Query parameters: {'experiment_language': 'Java', 'expertise_experiment_language': 'high'}
Filtered experiment IDs saved to E:\Storage\ETH\7. Semester\Thesis\Code\src\query\output\experiment_language\expertise_experiment_language\Java_high.csv


In [2]:
# query_output_path = r"D:\Storage\ETH\Thesis\Code\src\query\output\experiment_language\expertise_experiment_language"
query_output_path = r"E:\Storage\ETH\7. Semester\Thesis\Code\src\query\output\experiment_language\expertise_experiment_language"

Run within-group comparison

In [3]:
from tools.comparison import within_group_comparison
from tools import auxiliary

# Example usage
eye_events_corrected = auxiliary.parse_corrected_emip_data()

!cd ../../../ && python -m Code.src.tools.scripts.comparison --dataset EMIP_corrected --trial_id 2 --comparison_type within

!cd ../../../ && python -m Code.src.tools.scripts.comparison --dataset EMIP_corrected --trial_id 5 --comparison_type within

^C


usage: comparison.py [-h] --dataset DATASET --trial_id TRIAL_ID
                     [--comparison_type {within,between,both}]
                     [--workers WORKERS]
comparison.py: error: argument --trial_id: invalid int value: "'5'"


Parallel Multimatch Comparison Script
Dataset: EMIP_corrected
Trial ID: 2
Comparison Type: within
Concurrent Workers: 18

Loading eye event data...
Loaded 61799 eye event records

Running within-group comparison...
Processing group: Java_high.csv with 12 IDs with trial_id 2
  Total comparisons to perform: 132
Processing group: Java_low.csv with 48 IDs with trial_id 2
  Total comparisons to perform: 2256



  Java_high.csv: 100%|██████████| 132/132 [00:09<00:00, 13.77pair/s]

  Java_low.csv:   2%|▏         | 42/2256 [00:05<04:53,  7.56pair/s]
Process SpawnProcess-33:
Traceback (most recent call last):
  File "D:\Storage\ETH\Thesis\python\Lib\multiprocessing\process.py", line 314, in _bootstrap
    self.run()
  File "D:\Storage\ETH\Thesis\python\Lib\multiprocessing\process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "D:\Storage\ETH\Thesis\python\Lib\concurrent\futures\process.py", line 249, in _process_worker
    call_item = call_queue.get(block=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\Storage\ETH\Thesis\python\Lib\multiprocessing\queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
MemoryError
Traceback (most recent call last):
  File "D:\Storage\ETH\Thesis\python\Lib\multiprocessing\queues.py", line 246, in _feed
    send_bytes(obj)
  File "D:\Storage\ETH\Thesis\python\Lib\multiprocessi

Run between-group comparison

In [ ]:
from tools.comparison import between_group_comparison
from tools import auxiliary

eye_events_corrected = auxiliary.parse_corrected_emip_data()

!cd ../../../ && python -m Code.src.tools.scripts.comparison --dataset EMIP_corrected --trial_id 2 --comparison_type between
comparison_between_results = between_group_comparison(query_output_path, eye_events_corrected, trial_id='2', dataset='EMIP_corrected')

!cd ../../../ && python -m Code.src.tools.scripts.comparison --dataset EMIP_corrected --trial_id 5 --comparison_type between
comparison_between_results = between_group_comparison(query_output_path, eye_events_corrected, trial_id='5', dataset='EMIP_corrected')

Processing groups: Java_high.csv vs Java_low.csv with trial_id 2
Results for group pair Java_high.csv vs Java_low.csv saved to E:\Storage\ETH\7. Semester\Thesis\Code\output\processed_dataset\EMIP_corrected\between_group\trial_2\Java_high_Java_low_results.csv
Processing groups: Java_high.csv vs Java_medium.csv with trial_id 2
Results for group pair Java_high.csv vs Java_medium.csv saved to E:\Storage\ETH\7. Semester\Thesis\Code\output\processed_dataset\EMIP_corrected\between_group\trial_2\Java_high_Java_medium_results.csv
Processing groups: Java_high.csv vs Java_none.csv with trial_id 2
Results for group pair Java_high.csv vs Java_none.csv saved to E:\Storage\ETH\7. Semester\Thesis\Code\output\processed_dataset\EMIP_corrected\between_group\trial_2\Java_high_Java_none_results.csv
Processing groups: Java_low.csv vs Java_medium.csv with trial_id 2
Results for group pair Java_low.csv vs Java_medium.csv saved to E:\Storage\ETH\7. Semester\Thesis\Code\output\processed_dataset\EMIP_corrected\b

Merge csv data into one file

In [4]:
import os, glob
import pandas as pd
from tools.path import setup_paths

paths = setup_paths()

dirs = [
    os.path.join(paths["output_path"], "processed_dataset", "EMIP_corrected", "within_group", "trial_2"),
    os.path.join(paths["output_path"], "processed_dataset", "EMIP_corrected", "within_group", "trial_5"),
    os.path.join(paths["output_path"], "processed_dataset", "EMIP_corrected", "between_group", "trial_2"),
    os.path.join(paths["output_path"], "processed_dataset", "EMIP_corrected", "between_group", "trial_5")
]

def concatenate_csv_files():
    """
    Read all CSV files in the current folder, merge them, and save to the current folder
    """
    # Get the current folder path
    # current_folder = os.path.dirname(os.path.abspath(__file__))
    current_folder = dir

    # Find all CSV files in the current folder
    csv_files = glob.glob(os.path.join(current_folder, "*.csv"))
    
    if not csv_files:
        print("Current folder has no CSV files.")
        return

    print(f"Found {len(csv_files)} CSV files.")

    # Read and concatenate all CSV files
    dataframes = []
    for file in csv_files:
        try:
            df = pd.read_csv(file)
            # Optional: Add the filename as a column
            df['source_file'] = os.path.basename(file)
            dataframes.append(df)
            print(f"Already read: {os.path.basename(file)} ({len(df)} rows)")
        except Exception as e:
            print(f"Error reading {file}: {e}")

    if dataframes:
        # Combine all dataframes
        combined_df = pd.concat(dataframes, ignore_index=True)
        
        # Save the combined dataframe to a new CSV file
        output_file = os.path.join(current_folder, "combined_data.csv")
        combined_df.to_csv(output_file, index=False)

        print(f" Combination finished {len(combined_df)} rows")
        print(f"Results saved to: {output_file}")
    else:
        print("No CSV files were successfully read.")

# Run the function
if __name__ == "__main__":
    for dir in dirs:
        print(f"Processing directory: {dir}")
        os.chdir(dir)
        concatenate_csv_files()

Processing directory: D:\Storage\ETH\Thesis\Code\output\processed_dataset\EMIP_corrected\within_group\trial_2
Found 5 CSV files.
Already read: combined_data.csv (3356 rows)
Already read: Java_high_results.csv (55 rows)
Already read: Java_low_results.csv (903 rows)
Already read: Java_medium_results.csv (2145 rows)
Already read: Java_none_results.csv (253 rows)
 Combination finished 6712 rows
Results saved to: D:\Storage\ETH\Thesis\Code\output\processed_dataset\EMIP_corrected\within_group\trial_2\combined_data.csv
Processing directory: D:\Storage\ETH\Thesis\Code\output\processed_dataset\EMIP_corrected\within_group\trial_5
Found 5 CSV files.
Already read: combined_data.csv (2492 rows)
Already read: Java_high_results.csv (36 rows)
Already read: Java_low_results.csv (780 rows)
Already read: Java_medium_results.csv (1540 rows)
Already read: Java_none_results.csv (136 rows)
 Combination finished 4984 rows
Results saved to: D:\Storage\ETH\Thesis\Code\output\processed_dataset\EMIP_corrected\wit